In [147]:
import pandas as pd
import os
import numpy as np
from tqdm import tqdm

In [148]:
def id_sort(data):
    data['SortKey'] = data.iloc[:, 0].str.extract(r'(\d+)').astype(int)  # 提取數字部分，轉換為數值型
    data_sorted = data.sort_values(by='SortKey').reset_index(drop=True)  # 按數字部分排序
    data_sorted = data_sorted.drop(columns=['SortKey'])  # 刪除輔助列
    
    return data_sorted

In [149]:
file_path = r"V:\dunwei\MACE\dataset\pre_process\combine_feature/"
save_data_path = r"V:\dunwei\MACE\dataset\pre_process\combine_feature/"

In [150]:
if not os.path.exists(save_data_path):
    os.makedirs(save_data_path)
    print(f"Created directory: {save_data_path}")
else:
    print(f"{save_data_path} already exists.")

V:\dunwei\MACE\dataset\pre_process\combine_feature/ already exists.


In [151]:
mode = 2
"""
1: missing value
2: non-missing value
"""
if mode == 1:
    h_data = pd.read_csv(file_path + "missing_value/combine_h_missing.csv")
    p_data = pd.read_csv(file_path + "missing_value/combine_p_missing.csv")
elif mode == 2:
    h_data = pd.read_csv(file_path + "non_missing_value/combine_h_non_missing.csv")
    p_data = pd.read_csv(file_path + "non_missing_value/combine_p_non_missing.csv")


In [152]:
h_counts = h_data.iloc[:, 0].value_counts().reset_index()
h_counts = id_sort(h_counts)
display(h_counts)
p_counts = p_data.iloc[:, 0].value_counts().reset_index()
p_counts = id_sort(p_counts)
display(p_counts)

,Transformation_ID,count
0,MACE001,12549
1,MACE002,12549
2,MACE003,12549
3,MACE004,12549
4,MACE005,12549
...,...,...
427,MACE520,12549
428,MACE521,12549
429,MACE522,12549
430,MACE523,12549


,Transformation_ID,count
0,MACE021,12549
1,MACE028,12549
2,MACE033,12549
3,MACE044,12549
4,MACE046,12549
...,...,...
84,MACE503,12549
85,MACE505,12549
86,MACE507,12549
87,MACE508,12549


In [153]:
header = h_data.columns.tolist()
h_header = h_counts['Transformation_ID']
p_header = p_counts['Transformation_ID']
print(p_header)

0     MACE021
1     MACE028
2     MACE033
3     MACE044
4     MACE046
       ...   
84    MACE503
85    MACE505
86    MACE507
87    MACE508
88    MACE513
Name: Transformation_ID, Length: 89, dtype: object


In [154]:
h_data_avg = []
st_h = 0
for i in range(len(h_counts)):
    current_h = h_counts.iloc[i, 1]
    ed_h = st_h + current_h
    prob_avg = h_data.loc[st_h:ed_h, ['prob']].mean(axis=0).values.tolist()
    prob_avg = np.hstack((h_header[i], prob_avg))
    h_data_avg.append(prob_avg)
    st_h = ed_h
H_prob_avg = pd.DataFrame(h_data_avg, columns=(header[0],header[6]))
display(H_prob_avg)


,Transformation_ID,prob
0,MACE001,0.004271554791280069
1,MACE002,0.019440638568956017
2,MACE003,0.0046442139511268885
3,MACE004,0.007679609082526693
4,MACE005,0.008264200060358039
...,...,...
427,MACE520,0.005429194394544542
428,MACE521,0.004523684298890279
429,MACE522,0.005474426809562766
430,MACE523,0.009761869447944698


In [155]:
combine_h_avg = pd.DataFrame()
st_h = 0
current_h = 0
for i in range(len(h_counts)):
    st_h = st_h + current_h
    if h_data.iloc[st_h, 0] == H_prob_avg.iloc[i, 0]:
        current_h = h_counts.iloc[i, 1]
        data = pd.concat([h_data.iloc[[st_h],0:6].reset_index(drop=True), H_prob_avg.iloc[[i], 1].reset_index(drop=True), h_data.iloc[[st_h],[-1]].reset_index(drop=True)], axis = 1)
        combine_h_avg = pd.concat([combine_h_avg, data], axis=0, ignore_index=True)
display(combine_h_avg)
        

,Transformation_ID,count_v_ar,count_copd,smoke_try,count_dm,male,prob,mace
0,MACE001,0.0,0.0,1.0,0.0,1,0.004271554791280069,0
1,MACE002,0.0,0.0,1.0,1.0,1,0.019440638568956017,0
2,MACE003,0.0,0.0,0.0,0.0,0,0.0046442139511268885,0
3,MACE004,0.0,0.0,0.0,0.0,1,0.007679609082526693,0
4,MACE005,0.0,0.0,1.0,0.0,1,0.008264200060358039,0
...,...,...,...,...,...,...,...,...
427,MACE520,0.0,0.0,1.0,0.0,1,0.005429194394544542,0
428,MACE521,0.0,0.0,0.0,0.0,1,0.004523684298890279,0
429,MACE522,0.0,0.0,1.0,1.0,1,0.005474426809562766,0
430,MACE523,0.0,0.0,0.0,1.0,1,0.009761869447944698,0


In [156]:
p_data_avg = []
st_p = 0
for i in range(len(p_counts)):
    current_p = p_counts.iloc[i, 1]
    ed_p = st_p + current_p
    prob_avg = p_data.loc[st_p:ed_p, ['prob']].mean(axis=0).values.tolist()
    prob_avg = np.hstack((p_header[i], prob_avg))
    p_data_avg.append(prob_avg)
    st_p = ed_p
P_prob_avg = pd.DataFrame(p_data_avg, columns=(header[0],header[6]))
display(P_prob_avg)


,Transformation_ID,prob
0,MACE021,0.007354534270065752
1,MACE028,0.004482223426134403
2,MACE033,0.010779731103444212
3,MACE044,0.01897427040253896
4,MACE046,0.008259250293418162
...,...,...
84,MACE503,0.008971306479260182
85,MACE505,0.016154569979156812
86,MACE507,0.013308819637503735
87,MACE508,0.9999207784492988


In [157]:
combine_p_avg = pd.DataFrame()
st_p = 0
current_p = 0
for i in range(len(p_counts)):
    st_p = st_p + current_p
    if p_data.iloc[st_p, 0] == P_prob_avg.iloc[i, 0]:
        current_p = p_counts.iloc[i, 1]
        data = pd.concat([p_data.iloc[[st_p],0:6].reset_index(drop=True), P_prob_avg.iloc[[i], 1].reset_index(drop=True), p_data.iloc[[st_p],[-1]].reset_index(drop=True)], axis = 1)
        combine_p_avg = pd.concat([combine_p_avg, data], axis=0, ignore_index=True)
display(combine_p_avg)

,Transformation_ID,count_v_ar,count_copd,smoke_try,count_dm,male,prob,mace
0,MACE021,0.0,0.0,0.0,0.0,0,0.007354534270065752,1
1,MACE028,0.0,0.0,1.0,0.0,1,0.004482223426134403,1
2,MACE033,0.0,0.0,1.0,0.0,1,0.010779731103444212,1
3,MACE044,0.0,1.0,0.0,1.0,0,0.01897427040253896,1
4,MACE046,0.0,0.0,1.0,1.0,1,0.008259250293418162,1
...,...,...,...,...,...,...,...,...
84,MACE503,0.0,0.0,0.0,1.0,0,0.008971306479260182,1
85,MACE505,0.0,0.0,1.0,1.0,1,0.016154569979156812,1
86,MACE507,0.0,0.0,0.0,1.0,1,0.013308819637503735,1
87,MACE508,0.0,0.0,1.0,0.0,1,0.9999207784492988,1


In [158]:
if mode == 1:
    combine_h_avg.to_csv(save_data_path + "missing_value/combine_h_avg_missing_value.csv", index=False)
    combine_p_avg.to_csv(save_data_path + "missing_value/combine_p_avg_missing_value.csv", index=False)
elif mode == 2:
    combine_h_avg.to_csv(save_data_path + "non_missing_value/combine_h_avg_non_missing_value.csv", index=False)
    combine_p_avg.to_csv(save_data_path + "non_missing_value/combine_p_avg_non_missing_value.csv", index=False)